# Total model fine-tuning notebook

Three-stage training in the familiar notebook layout.
The supplied vision and GT-pose checkpoints are loaded once. The first
`ADAPT_EPOCHS` adapt pose with frozen/eval-mode vision, then joint training updates
all modules until `FREEZE_VISION_EPOCH`. The final phase freezes vision and
correspondence modules and trains pose only. `NUM_EPOCHS` includes all stages.

Pose training starts with GT cylinder matches. During joint training, the share of predicted matches ramps up to 100% by `FREEZE_VISION_EPOCH`. The frozen final phase uses only
predicted matches. Validation reports pose loss and valid-pose coverage separately for GT and predicted matches.


In [15]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [16]:
%reload_ext autoreload
%autoreload 2

import argparse
import json
import os
import random
import numpy as np
import time
import torch
from torch.utils.data import DataLoader
from dataset import SceneTwoPairsDataset
from forest_dataset import ForestPairDataset
from augmentations import build_train_transform, build_eval_transform
from model import PairImageCylinderModel
from metrics import relative_cylinder_errors
from losses import (
    vision_loss,
    patch_correspondence_loss,
    relative_depth_structure_loss,
    matched_depth_delta_loss,
    masked_relative_pose_loss_2d,
    matched_reprojection_loss_2d
)
from pathlib import Path
import matplotlib.pyplot as plt
from utils import plot_estimated_cylinders_on_images, plot_relative_pose, pose_to_text


In [17]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DATA_DIR = Path(".")
OUTPUT_DIR = Path("runs/total_finetune_224_skrylle")
#VISION_CHECKPOINT = Path("runs/vision_geometry_org_new_corr_new_data/checkpoints/best_vision.pt")
VISION_CHECKPOINT = Path("runs/vision_semi_good/checkpoints/latest.pt")
POSE_CHECKPOINT = Path("arrhenius_runs/slurm_pose_only_5842490/checkpoints/best.pt")

BATCH_SIZE = 8
NUM_EPOCHS = 200  # Total epochs; vision freezes after FREEZE_VISION_EPOCH.
ADAPT_EPOCHS = 10
FREEZE_VISION_EPOCH = 50  # Epochs 1–50 train vision; epoch 51 onward is pose-only.

PRED_MATCH_RAMP_EPOCHS = FREEZE_VISION_EPOCH - ADAPT_EPOCHS
PRED_MATCH_MAX_PROB = 1.0  # Reach 100% predicted matches by the freeze phase.

ADAPT_LEARNING_RATE = 1e-4
LEARNING_RATE = 1e-5  # Vision LR during joint fine-tuning.
POSE_LEARNING_RATE = 1e-5  # Pose LR during joint fine-tuning.

VAL_INTERVAL = 10
SEED = 0

NUM_WORKERS = 4
IMG_SIZE = 224
PATCH_SIZE = (16, 4)
NUM_BINS = 128
EMBED_DIM = 192
DEPTH = 4
NUM_HEADS = 4
DROPOUT = 0.1
 
LAMBDA_OCC = 10.0
LAMBDA_RADIUS = 10.0
OCC_THRESH = 0.5
LAMBDA_CORR = 1.0
LAMBDA_POSE = 2.0
LAMBDA_DEPTH_DELTA = 0.35
LAMBDA_WIDTH = 0.1
LAMBDA_SINKHORN = 0.1

BEST_MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "best.pt")
LATEST_MODEL_PATH = str(OUTPUT_DIR / "checkpoints" / "latest.pt")
RESUME_TRAINING = False
RESUME_CHECKPOINT_PATH = LATEST_MODEL_PATH

BEST_ADAPT_PATH = str(OUTPUT_DIR / "checkpoints" / "best_adapt.pt")
BEST_POSE_PATH = str(OUTPUT_DIR / "checkpoints" / "best_pose.pt")
INFERENCE_MODEL = "best_pose"  # "best", "best_pose", "latest", or "best_adapt"
INFERENCE_BATCH_SIZE = 4
INFERENCE_SHOW_TEXT = False
INFERENCE_SHOW_IMAGES = True
VISION_SAMPLE_IDX = 0
FOREST_ENABLED = True
FOREST_START_EPOCH = 1
FOREST_BATCH_SIZE = 2
FOREST_SAMPLE_IDX = 0
FOREST_OCC_THRESHOLD = 0.3

# Default keeps the existing training/evaluation data unchanged.
# Set FOREST_DATASET_PRESET to one of the Skrylle entries to use that stride
# for the forest batches used during training.
FOREST_DATASET_PRESET = "skrylle_stride_20"

FOREST_DATASET_PRESETS = {
    "torpa": (DATA_DIR / "forest_torpa", "relative_poses.json"),
    "skrylle_stride_1": (DATA_DIR / "forestdataset" / "skrylle", "relative_poses_stride_1.json"),
    "skrylle_stride_5": (DATA_DIR / "forestdataset" / "skrylle", "relative_poses_stride_5.json"),
    "skrylle_stride_10": (DATA_DIR / "forestdataset" / "skrylle", "relative_poses_stride_10.json"),
    "skrylle_stride_20": (DATA_DIR / "forestdataset" / "skrylle", "relative_poses_stride_20.json"),
}

# Evaluate these independently in the forest evaluation cell. To compare all
# Skrylle baselines, use all four skrylle_stride_* names here.
FOREST_EVAL_PRESETS = ["snoge_stride_*"]



In [18]:
# Check component settings, configure device and seeds (same as train_tot.py).
parser = argparse.ArgumentParser()
if not 0 <= ADAPT_EPOCHS < FREEZE_VISION_EPOCH < NUM_EPOCHS:
    parser.error("Require 0 <= --adapt-epochs < --freeze-vision-epoch < --epochs")

if min(LEARNING_RATE, ADAPT_LEARNING_RATE, LAMBDA_POSE) <= 0:
    parser.error("Learning rates and --lambda-pose must be positive")

POSE_LEARNING_RATE = LEARNING_RATE if POSE_LEARNING_RATE is None else POSE_LEARNING_RATE

if POSE_LEARNING_RATE <= 0 or min(VAL_INTERVAL, BATCH_SIZE) < 1:
    parser.error("Pose learning rate, intervals and batch size must be positive")

pose_saved = torch.load(POSE_CHECKPOINT, map_location="cpu", weights_only=False)

if "model_config" not in pose_saved:
    parser.error("--pose-checkpoint must be the standalone matched-pair pose checkpoint")

pose_cfg = pose_saved["model_config"]

POSE_EMBED_DIM = pose_cfg["embed_dim"]
POSE_NUM_HEADS = pose_cfg["num_heads"]
POSE_DEPTH = pose_cfg["depth"]
POSE_DROPOUT = pose_cfg["dropout"]
FOV_DEGREES = pose_cfg["fov_degrees"]
RADIUS_SCALE = pose_cfg["radius_scale"]
DISTANCE_SCALE = pose_cfg["distance_scale"]

vision_saved = torch.load(VISION_CHECKPOINT, map_location="cpu", weights_only=False)
vision_cfg = vision_saved.get("args", {})

for key in ("img_size", "patch_size", "embed_dim", "depth", "num_heads", "num_bins", "dropout"):
    if key in vision_cfg:
        saved_value, requested_value = (vision_cfg[key], globals()[key.upper()])
        if key == "patch_size":
            saved_value, requested_value = (tuple(saved_value), tuple(requested_value))
        if saved_value != requested_value:
            parser.error(
                f"Vision checkpoint {key}={saved_value}, but script uses {requested_value}; "
                f"set --{key.replace('_', '-')} to match"
            )

if abs(vision_cfg.get("fov_degrees", 90.0) - FOV_DEGREES) > 1e-06:
    parser.error("Vision and pose checkpoints must use the same field of view")

del vision_saved

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_float32_matmul_precision("high")

if device.type == "cuda":
    torch.backends.cudnn.benchmark = False

os.makedirs(OUTPUT_DIR, exist_ok=True)

os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Seed: ", SEED)


Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
CUDA: 13.0
Seed:  0


In [19]:
# Fresh runs initialize from the vision and GT-pose checkpoints. For continuation, set RESUME_TRAINING=True.
print("Vision checkpoint:", VISION_CHECKPOINT)
print("GT-pose checkpoint:", POSE_CHECKPOINT)
print(f"Resume training: {RESUME_TRAINING}; checkpoint: {RESUME_CHECKPOINT_PATH}")
print(f"Pose adaptation through epoch {ADAPT_EPOCHS}; joint training through epoch {FREEZE_VISION_EPOCH}; pose-only refinement through epoch {NUM_EPOCHS}.")


Vision checkpoint: runs/vision_semi_good/checkpoints/latest.pt
GT-pose checkpoint: arrhenius_runs/slurm_pose_only_5842490/checkpoints/best.pt
Resume training: False; checkpoint: runs/total_finetune_224_skrylle/checkpoints/latest.pt
Pose adaptation through epoch 10; joint training through epoch 50; pose-only refinement through epoch 200.


## Data

Data loaders and the composed model are initialized in the same order as
`train_tot.py`. The forest loader supplies unlabeled image pairs for Sinkhorn reprojection loss during training.


In [20]:
train_dataset = SceneTwoPairsDataset(
    root_dir=os.path.join(DATA_DIR, "dataset"),
    image_size=IMG_SIZE,
    debug=False,
    return_two_pairs=False,
    transform=build_train_transform(IMG_SIZE),
)

val_dataset = SceneTwoPairsDataset(
    root_dir=os.path.join(DATA_DIR, "valdataset"),
    image_size=IMG_SIZE,
    debug=False,
    return_two_pairs=False,
    transform=build_eval_transform(IMG_SIZE),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

model = PairImageCylinderModel(
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    in_chans=3,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    num_bins=NUM_BINS,
    dropout=DROPOUT,
    pose_embed_dim=POSE_EMBED_DIM,
    pose_num_heads=POSE_NUM_HEADS,
    pose_depth=POSE_DEPTH,
    pose_dropout=POSE_DROPOUT,
    fov_degrees=FOV_DEGREES,
    radius_scale=RADIUS_SCALE,
    distance_scale=DISTANCE_SCALE,
).to(device)

model.load_vision_checkpoint(VISION_CHECKPOINT)

model.pose_head.load_state_dict(pose_saved["model_state_dict"], strict=True)

print("Loaded vision weights:", VISION_CHECKPOINT)
print("Loaded GT-pose weights:", POSE_CHECKPOINT)

del pose_saved

n_params = sum((p.numel() for p in model.parameters()))

print(f"Model parameters: {n_params / 1000000.0:.2f} M")

raw_model = model

vision_parameters = (
    list(model.backbone.parameters()) + list(model.vision_head.parameters()) + list(model.corr_decoder.parameters())
)

optimizer = torch.optim.AdamW(
    [
        {"params": vision_parameters, "lr": LEARNING_RATE, "name": "vision"},
        {"params": list(model.pose_head.parameters()), "lr": ADAPT_LEARNING_RATE, "name": "pose"},
    ],
    weight_decay=0.05,
)

#scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=False)

resume_checkpoint = None
START_EPOCH = 0
if RESUME_TRAINING:
    resume_path = Path(RESUME_CHECKPOINT_PATH)
    if not resume_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
    resume_checkpoint = torch.load(resume_path, map_location=device, weights_only=False)
    for key in ("model_state_dict", "optimizer_state_dict", "epoch", "history"):
        if key not in resume_checkpoint:
            raise ValueError(f"Checkpoint is missing required resume field: {key}")
    raw_model.load_state_dict(resume_checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in resume_checkpoint:
        scaler.load_state_dict(resume_checkpoint["scaler_state_dict"])
    START_EPOCH = int(resume_checkpoint["epoch"])
    rng_state = resume_checkpoint.get("rng_state")
    if rng_state is not None:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"].cpu())
        if device.type == "cuda" and rng_state.get("cuda") is not None:
            torch.cuda.set_rng_state_all([state.cpu() for state in rng_state["cuda"]])
    if START_EPOCH >= NUM_EPOCHS:
        raise ValueError(f"Checkpoint is already at epoch {START_EPOCH}; NUM_EPOCHS is {NUM_EPOCHS}.")
    print(f"Resuming from epoch {START_EPOCH}; next epoch is {START_EPOCH + 1}.")
else:
    print("Starting from the supplied vision and GT-pose checkpoints.")

forest_train_loader = None
if FOREST_ENABLED:
    if FOREST_DATASET_PRESET not in FOREST_DATASET_PRESETS:
        raise ValueError(f"Unknown FOREST_DATASET_PRESET: {FOREST_DATASET_PRESET}")
    forest_train_dir, forest_train_poses = FOREST_DATASET_PRESETS[FOREST_DATASET_PRESET]
    forest_train_dataset = ForestPairDataset(
        dataset_dir=forest_train_dir,
        poses_file=forest_train_poses,
        transform=build_eval_transform(IMG_SIZE),
    )
    forest_train_loader = DataLoader(
        forest_train_dataset, batch_size=FOREST_BATCH_SIZE, shuffle=True, num_workers=0,
        pin_memory=device.type == "cuda", drop_last=True,
    )
    if len(forest_train_dataset) < FOREST_BATCH_SIZE:
        raise ValueError("Forest dataset must contain at least FOREST_BATCH_SIZE pairs.")
    print(
        f"Forest training data: {FOREST_DATASET_PRESET} "
        f"({forest_train_dir / forest_train_poses}); "
        f"samples: {len(forest_train_dataset)}; batch size: {FOREST_BATCH_SIZE}"
    )


Train samples: 4794
Validation samples: 482
Train batches: 600
Validation batches: 61
Loaded vision weights: runs/vision_semi_good/checkpoints/latest.pt
Loaded GT-pose weights: arrhenius_runs/slurm_pose_only_5842490/checkpoints/best.pt
Model parameters: 4.46 M
Starting from the supplied vision and GT-pose checkpoints.
Forest training data: skrylle_stride_20 (forestdataset/skrylle/relative_poses_stride_20.json); samples: 955; batch size: 2


In [21]:
if False:
    from PIL import Image
    import matplotlib.pyplot as plt
    import torch

    # Ta första bilden från första träningsparet
    sample = train_dataset.samples[0]
    image_path = train_dataset._resolve_path(
        sample.scene_dir, sample.pair["image1"]
    )
    with Image.open(image_path) as im:
        original = im.convert("RGB")

    # Varje anrop ger ett nytt slumpmässigt augmenterat utfall
    augmented = [build_train_transform(IMG_SIZE)(original.copy()) for _ in range(7)]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    images = [build_eval_transform(IMG_SIZE)(original)] + augmented
    titles = ["Original"] + [f"Augmentering {i}" for i in range(1, 8)]

    for ax, image, title in zip(axes.flat, images, titles):
        ax.imshow(image.permute(1, 2, 0).clamp(0, 1))
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Training helpers

Batch transfer and freeze/unfreeze behavior are the same as in `train_tot.py`.


In [22]:
def move_batch_to_device(batch, device):
    return tuple((x.to(device, non_blocking=True) for x in batch))


def set_training_stage(model, optimizer, pose_only, freeze_vision=False):
    """Freeze vision for adaptation or final pose-only refinement."""
    freeze_vision_modules = pose_only or freeze_vision
    for module in (model.backbone, model.vision_head, model.corr_decoder):
        module.requires_grad_(not freeze_vision_modules)
    model.pose_head.requires_grad_(True)
    for group in optimizer.param_groups:
        if group["name"] == "vision":
            group["lr"] = 0.0 if freeze_vision_modules else LEARNING_RATE
        elif group["name"] == "pose":
            group["lr"] = ADAPT_LEARNING_RATE if pose_only else POSE_LEARNING_RATE


In [23]:
@torch.no_grad()
def validate(
    model,
    loader,
    device,
    occ_thresh,
    lambda_occ,
    lambda_radius,
    lambda_corr,
    lambda_depth_delta,
    lambda_pose,
    lambda_width,
    lambda_sinkhorn=0.0,
    pose_only=False,
    matching_mode="predicted",
):
    model.eval()
    totals = {
        "total": torch.zeros((), device=device),
        "vision": torch.zeros((), device=device),
        "vision_occupancy": torch.zeros((), device=device),
        "vision_radius": torch.zeros((), device=device),
        "vision_depth": torch.zeros((), device=device),
        "vision_width": torch.zeros((), device=device),
        "objective": torch.zeros((), device=device),
        "correspondence": torch.zeros((), device=device),
        "depth_structure": torch.zeros((), device=device),
        "depth_delta": torch.zeros((), device=device),
        "sinkhorn_reprojection": torch.zeros((), device=device),
        "pose": torch.zeros((), device=device),
        "translation": torch.zeros((), device=device),
        "rotation": torch.zeros((), device=device),
        "cylinder_radius_rel_l2": torch.zeros((), device=device),
        "cylinder_position_rel_l2": torch.zeros((), device=device),
        "pose_valid_samples": torch.zeros((), device=device),
        "pose_samples": torch.zeros((), device=device),
    }
    num_batches = 0
    total_sinkhorn_matches = torch.zeros((), device=device)
    for batch in loader:
        img_a, vision_a, img_b, vision_b, pose_ab = move_batch_to_device(batch, device)
        pred_vision_a, pred_vision_b, pred_pose, corr_a, corr_b, pose_info = model(
            img_a,
            img_b,
            return_corr=True,
            compute_pose=lambda_pose != 0,
            target_vision_a=vision_a,
            target_vision_b=vision_b,
            return_pose_inputs=True,
            occ_thresh=occ_thresh,
            matching_mode=matching_mode,
        )

        vis_a, occ_a, rad_a, dep_a, width_a = vision_loss(
            pred_vision_a,
            vision_a,
            occ_thresh=occ_thresh,
            lambda_occ=lambda_occ,
            lambda_radius=lambda_radius,
            lambda_depth=1.0,
            lambda_width=lambda_width,
        )
        vis_b, occ_b, rad_b, dep_b, width_b = vision_loss(
            pred_vision_b,
            vision_b,
            occ_thresh=occ_thresh,
            lambda_occ=lambda_occ,
            lambda_radius=lambda_radius,
            lambda_depth=1.0,
            lambda_width=lambda_width,
        )
        vision_loss_total = 0.5 * (vis_a + vis_b)

        corr_loss = patch_correspondence_loss(corr_a, vision_a, corr_b, vision_b)
        
        depth_structure_loss = 0.5 * (
            relative_depth_structure_loss(pred_vision_a, vision_a, occ_thresh)
            + relative_depth_structure_loss(pred_vision_b, vision_b, occ_thresh)
        )
        depth_delta_loss = matched_depth_delta_loss(pred_vision_a, vision_a, pred_vision_b, vision_b, occ_thresh)
        
        if lambda_pose != 0 and lambda_sinkhorn > 0:
            sinkhorn_reproj_loss, sinkhorn_match_count = matched_reprojection_loss_2d(
                pred_vision_a.float(),
                pred_vision_b=pred_vision_b.float(),
                relative_pose_pred=pred_pose.float(),
                occ_thresh=occ_thresh,
                fov_degrees=FOV_DEGREES,
                matching_mode="sinkhorn",
                return_match_count=True,
            )
            total_sinkhorn_matches += sinkhorn_match_count
        else:
            sinkhorn_reproj_loss = vision_loss_total.new_zeros(())

        if lambda_pose != 0:
            pose_total, t_loss, r_loss = masked_relative_pose_loss_2d(
                pred_pose, pose_ab, pose_info["pose_valid"], t_weight=1.0, r_weight=1.0
            )
        else:
            pose_total = vision_loss_total.new_zeros(())
            t_loss = vision_loss_total.new_zeros(())
            r_loss = vision_loss_total.new_zeros(())

        total_loss = (
            vision_loss_total
            + lambda_corr * corr_loss
            + 0.5 * depth_structure_loss
            + lambda_depth_delta * depth_delta_loss
            + lambda_pose * pose_total
            + lambda_sinkhorn * sinkhorn_reproj_loss
        )
        radius_a, position_a = relative_cylinder_errors(pred_vision_a, vision_a, occ_thresh)
        radius_b, position_b = relative_cylinder_errors(pred_vision_b, vision_b, occ_thresh)
        objective = lambda_pose * pose_total if pose_only else total_loss

        totals["objective"] += objective.detach()
        totals["total"] += total_loss.detach()
        totals["vision"] += vision_loss_total.detach()
        totals["vision_occupancy"] += (0.5 * (occ_a + occ_b)).detach()
        totals["vision_radius"] += (0.5 * (rad_a + rad_b)).detach()
        totals["vision_depth"] += (0.5 * (dep_a + dep_b)).detach()
        totals["vision_width"] += (0.5 * (width_a + width_b)).detach()
        totals["correspondence"] += corr_loss.detach()
        totals["depth_structure"] += depth_structure_loss.detach()
        totals["depth_delta"] += depth_delta_loss.detach()
        totals["sinkhorn_reprojection"] += sinkhorn_reproj_loss.detach()
        totals["pose"] += pose_total.detach()
        totals["pose_valid_samples"] += pose_info["pose_valid"].sum().detach()
        totals["pose_samples"] += pose_ab.shape[0]
        totals["translation"] += t_loss.detach()
        totals["rotation"] += r_loss.detach()
        totals["cylinder_radius_rel_l2"] += (0.5 * (radius_a + radius_b)).detach()
        totals["cylinder_position_rel_l2"] += (0.5 * (position_a + position_b)).detach()

        num_batches += 1

    if num_batches == 0:
        raise RuntimeError("Validation loader is empty.")
    metric_keys = list(totals)
    metric_values = torch.stack([totals[key] / num_batches for key in metric_keys] + [total_sinkhorn_matches]).cpu().tolist()
    metrics = dict(zip(metric_keys, metric_values))
    metrics["sinkhorn_matches_per_sample"] = metric_values[-1] / max(metrics["pose_samples"], 1.0)
    metrics["pose_valid_fraction"] = metrics["pose_valid_samples"] / max(metrics["pose_samples"], 1.0)
    return metrics


## Training

In [24]:
metric_names = ("vision_occupancy", "vision_radius", "vision_depth", "vision_width")
history = {
    key: []
    for key in (
        "epoch",
        "stage",
        "vision_lr",
        "pose_lr",
        "pred_match_probability",
        "train_pred_match_batch_fraction",
        "objective",
        "loss",
        "vision_loss",
        "correspondence_loss",
        "depth_structure_loss",
        "depth_delta_loss",
        "sinkhorn_reprojection_loss",
        "forest_sinkhorn_loss",
        "sinkhorn_matches_per_sample",
        "pose_loss",
        "translation_loss",
        "rotation_loss",
        "val_epoch",
        "val_stage",
        "val_objective",
        "val_total",
        "val_vision",
        "val_correspondence",
        "val_depth_structure",
        "val_depth_delta",
        "val_sinkhorn_reprojection",
        "val_sinkhorn_matches_per_sample",
        "val_pose",
        "val_gt_pose",
        "val_pred_pose",
        "val_gt_pose_valid_fraction",
        "val_pred_pose_valid_fraction",
        "val_translation",
        "val_rotation",
    )
}
weights = dict(
    vision_occupancy=LAMBDA_OCC, vision_radius=LAMBDA_RADIUS, vision_depth=1.0, vision_width=LAMBDA_WIDTH
)
for component in metric_names:
    for prefix in ("", "val_"):
        history[prefix + component] = []
        history[prefix + component + "_weighted"] = []

best_val_loss = float("inf")
best_adapt_pose = float("inf")
best_pose_loss = float("inf")

if resume_checkpoint is not None:
    history = resume_checkpoint["history"]
    required_history = (
        "epoch", "pred_match_probability", "train_pred_match_batch_fraction",
        "val_epoch", "val_gt_pose", "val_pred_pose",
        "val_gt_pose_valid_fraction", "val_pred_pose_valid_fraction",
    )
    missing_history = [key for key in required_history if key not in history]
    if missing_history:
        raise ValueError(
            "Resume checkpoint is incompatible with this notebook version; missing history keys: "
            + ", ".join(missing_history)
        )
    best_val_loss = resume_checkpoint.get("best_val_loss", float("inf"))
    best_adapt_pose = resume_checkpoint.get("best_adapt_pose", float("inf"))
    best_pose_loss = resume_checkpoint.get(
        "best_pose_loss", min(history["val_pred_pose"], default=float("inf"))
    )
    print(f"Restored history through epoch {START_EPOCH}.")

history.setdefault("forest_sinkhorn_loss", [float("nan")] * len(history.get("epoch", [])))
history.setdefault("sinkhorn_matches_per_sample", [float("nan")] * len(history.get("epoch", [])))
history.setdefault("val_sinkhorn_matches_per_sample", [float("nan")] * len(history.get("val_epoch", [])))

previous_stage = None
forest_iterator = None

for epoch in range(START_EPOCH + 1, NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    pose_only = epoch <= ADAPT_EPOCHS
    freeze_vision = epoch > FREEZE_VISION_EPOCH
    stage = "pose_adapt" if pose_only else ("pose_refine" if freeze_vision else "joint")
    pose_objective_only = pose_only or freeze_vision
    joint_epochs_elapsed = max(0, epoch - ADAPT_EPOCHS)
    ramp_fraction = min(joint_epochs_elapsed / max(PRED_MATCH_RAMP_EPOCHS, 1), 1.0)
    pred_match_probability = (
        0.0 if pose_only else (1.0 if freeze_vision else PRED_MATCH_MAX_PROB * ramp_fraction)
    )
    set_training_stage(model, optimizer, pose_only, freeze_vision=freeze_vision)
    if stage != previous_stage:
        print(
            f"\nStage: {stage}; vision lr={optimizer.param_groups[0]['lr']:g}; "
            f"pose lr={optimizer.param_groups[1]['lr']:g}",
            flush=True,
        )
        previous_stage = stage
    model.train()
    if pose_objective_only:
        model.backbone.eval()
        model.vision_head.eval()
        model.corr_decoder.eval()
    pred_match_batches = 0
    if FOREST_ENABLED and epoch >= FOREST_START_EPOCH and forest_iterator is None:
        forest_iterator = iter(forest_train_loader)
    train_totals = {
        "total": torch.zeros((), device=device),
        "vision": torch.zeros((), device=device),
        "vision_occupancy": torch.zeros((), device=device),
        "vision_radius": torch.zeros((), device=device),
        "vision_depth": torch.zeros((), device=device),
        "vision_width": torch.zeros((), device=device),
        "objective": torch.zeros((), device=device),
        "correspondence": torch.zeros((), device=device),
        "depth_structure": torch.zeros((), device=device),
        "depth_delta": torch.zeros((), device=device),
        "sinkhorn_reprojection": torch.zeros((), device=device),
        "sinkhorn_match_count": torch.zeros((), device=device),
        "forest_sinkhorn": torch.zeros((), device=device),
        "pose": torch.zeros((), device=device),
        "translation": torch.zeros((), device=device),
        "rotation": torch.zeros((), device=device),
    }
    for batch in train_loader:
        matching_mode = "predicted" if random.random() < pred_match_probability else "gt"
        pred_match_batches += int(matching_mode == "predicted")
        img_a, vision_a, img_b, vision_b, pose_ab = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        forest_batch = None
        if FOREST_ENABLED and epoch >= FOREST_START_EPOCH:
            try:
                forest_batch = next(forest_iterator)
            except StopIteration:
                forest_iterator = iter(forest_train_loader)
                forest_batch = next(forest_iterator)
            img_a_forest, img_b_forest, _ = move_batch_to_device(forest_batch, device)

        #with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        with torch.autocast(device_type=device.type, enabled=False):
            model_outputs = model(
                img_a,
                img_b,
                return_corr=not pose_objective_only,
                compute_pose=LAMBDA_POSE != 0,
                target_vision_a=vision_a,
                target_vision_b=vision_b,
                return_pose_inputs=True,
                occ_thresh=OCC_THRESH,
                matching_mode=matching_mode,
            )
            if pose_objective_only:
                pred_vision_a, pred_vision_b, pred_pose, pose_info = model_outputs
                corr_a = corr_b = None
            else:
                pred_vision_a, pred_vision_b, pred_pose, corr_a, corr_b, pose_info = model_outputs
            if pose_objective_only:
                # These batches optimize pose only; skip the unused vision losses.
                if LAMBDA_POSE != 0:
                    pose_total, t_loss, r_loss = masked_relative_pose_loss_2d(
                        pred_pose, pose_ab, pose_info["pose_valid"], t_weight=1.0, r_weight=1.0
                    )
                else:
                    pose_total = pred_vision_a.new_zeros(())
                    t_loss = pred_vision_a.new_zeros(())
                    r_loss = pred_vision_a.new_zeros(())
                loss = LAMBDA_POSE * pose_total
            else:
                vis_a, occ_a, rad_a, dep_a, width_a = vision_loss(
                    pred_vision_a,
                    vision_a,
                    occ_thresh=OCC_THRESH,
                    lambda_occ=LAMBDA_OCC,
                    lambda_radius=LAMBDA_RADIUS,
                    lambda_depth=1.0,
                    lambda_width=LAMBDA_WIDTH,
                )
                vis_b, occ_b, rad_b, dep_b, width_b = vision_loss(
                    pred_vision_b,
                    vision_b,
                    occ_thresh=OCC_THRESH,
                    lambda_occ=LAMBDA_OCC,
                    lambda_radius=LAMBDA_RADIUS,
                    lambda_depth=1.0,
                    lambda_width=LAMBDA_WIDTH,
                )
                vision_loss_total = 0.5 * (vis_a + vis_b)

                corr_loss = patch_correspondence_loss(corr_a, vision_a, corr_b, vision_b)

                depth_structure_loss = 0.5 * (
                    relative_depth_structure_loss(pred_vision_a, vision_a, OCC_THRESH)
                    + relative_depth_structure_loss(pred_vision_b, vision_b, OCC_THRESH)
                )
                depth_delta_loss = matched_depth_delta_loss(
                    pred_vision_a, vision_a, pred_vision_b, vision_b, OCC_THRESH
                )
                
                if LAMBDA_POSE != 0:
                    pose_total, t_loss, r_loss = masked_relative_pose_loss_2d(
                        pred_pose, pose_ab, pose_info["pose_valid"], t_weight=1.0, r_weight=1.0
                    )
                else:
                    pose_total = pred_vision_a.new_zeros(())
                    t_loss = pred_vision_a.new_zeros(())
                    r_loss = pred_vision_a.new_zeros(())
                # Sinkhorn reprojection is applied to forest pairs below only.
                sinkhorn_reproj_loss = pred_vision_a.new_zeros(())
                sinkhorn_match_count = 0
                
                loss = (
                    vision_loss_total
                    + LAMBDA_CORR * corr_loss
                    + 0.5 * depth_structure_loss
                    + LAMBDA_DEPTH_DELTA * depth_delta_loss
                    + LAMBDA_POSE * pose_total
                    + LAMBDA_SINKHORN * sinkhorn_reproj_loss
                )
            forest_sinkhorn_loss = pred_vision_a.new_zeros(())
            if forest_batch is not None:
                forest_pred_a, forest_pred_b, forest_pose = model(
                    img_a_forest, img_b_forest, compute_pose=True,
                    matching_mode="predicted", occ_thresh=OCC_THRESH,
                )
                forest_sinkhorn_loss = matched_reprojection_loss_2d(
                    forest_pred_a.float(), pred_vision_b=forest_pred_b.float(),
                    relative_pose_pred=forest_pose.float(), occ_thresh=OCC_THRESH,
                    fov_degrees=FOV_DEGREES, matching_mode="sinkhorn",
                )
                loss = loss + LAMBDA_SINKHORN * forest_sinkhorn_loss
        objective = LAMBDA_POSE * pose_total if pose_objective_only else loss
        if forest_batch is not None and pose_objective_only:
            objective = objective + LAMBDA_SINKHORN * forest_sinkhorn_loss
        valid_pose = pose_info["pose_valid"] & (pose_ab[:, :2].norm(dim=-1) > 1e-06)
        
        if not pose_objective_only or valid_pose.any() or forest_batch is not None:
            scaler.scale(objective).backward()
            scaler.step(optimizer)
            scaler.update()
        
        train_totals["objective"] += objective.detach()
        if not pose_objective_only:
            train_totals["total"] += loss.detach()
            train_totals["vision"] += vision_loss_total.detach()
            train_totals["vision_occupancy"] += (0.5 * (occ_a + occ_b)).detach()
            train_totals["vision_radius"] += (0.5 * (rad_a + rad_b)).detach()
            train_totals["vision_depth"] += (0.5 * (dep_a + dep_b)).detach()
            train_totals["vision_width"] += (0.5 * (width_a + width_b)).detach()
            train_totals["correspondence"] += corr_loss.detach()
            train_totals["depth_structure"] += depth_structure_loss.detach()
            train_totals["depth_delta"] += depth_delta_loss.detach()
            train_totals["sinkhorn_reprojection"] += sinkhorn_reproj_loss.detach()
            train_totals["sinkhorn_match_count"] += sinkhorn_match_count
        train_totals["forest_sinkhorn"] += forest_sinkhorn_loss.detach()
        train_totals["pose"] += pose_total.detach()
        train_totals["translation"] += t_loss.detach()
        train_totals["rotation"] += r_loss.detach()
    
    metric_keys = list(train_totals)
    metric_values = torch.stack([train_totals[key] / len(train_loader) for key in metric_keys]).cpu().tolist()
    train_metrics = dict(zip(metric_keys, metric_values))
    sinkhorn_match_count = train_metrics.pop("sinkhorn_match_count")
    train_metrics["sinkhorn_matches_per_sample"] = (
        sinkhorn_match_count / max(len(train_loader.dataset), 1)
    )
    
    if pose_objective_only:
        for key in ('total', 'vision', 'vision_occupancy', 'vision_radius', 'vision_depth', 'vision_width', 'correspondence', 'depth_structure', 'depth_delta', 'sinkhorn_reprojection', 'sinkhorn_matches_per_sample'):
            train_metrics[key] = float("nan")
    history["epoch"].append(epoch)
    history["stage"].append(stage)
    history["vision_lr"].append(optimizer.param_groups[0]["lr"])
    history["pose_lr"].append(optimizer.param_groups[1]["lr"])
    history["pred_match_probability"].append(pred_match_probability)
    history["train_pred_match_batch_fraction"].append(pred_match_batches / max(len(train_loader), 1))
    train_mapping = {
        "objective": "objective",
        "loss": "total",
        "vision_loss": "vision",
        "correspondence_loss": "correspondence",
        "depth_structure_loss": "depth_structure",
        "depth_delta_loss": "depth_delta",
        "sinkhorn_reprojection_loss": "sinkhorn_reprojection",
        "forest_sinkhorn_loss": "forest_sinkhorn",
        "sinkhorn_matches_per_sample": "sinkhorn_matches_per_sample",
        "pose_loss": "pose",
        "translation_loss": "translation",
        "rotation_loss": "rotation",
    }
    for key, metric in train_mapping.items():
        history[key].append(train_metrics[metric])
    for component, weight in weights.items():
        history[component].append(train_metrics[component])
        history[component + "_weighted"].append(weight * train_metrics[component])
    
    train_loss_log = (
        f" | tot={train_metrics['total']:.4f} | vis={train_metrics['vision']:.4f} "
        f"| occ={train_metrics['vision_occupancy']:.4f} | rad={train_metrics['vision_radius']:.4f} "
        f"| depth={train_metrics['vision_depth']:.4f} | width={train_metrics['vision_width']:.4f} "
        f"| corr={train_metrics['correspondence']:.4f} | struct={train_metrics['depth_structure']:.4f} "
        f"| delta={train_metrics['depth_delta']:.4f}"
        if not pose_objective_only else ""
    )
    log = (
        f"Epoch {epoch} [{stage}]: obj={train_metrics['objective']:.4f}"
        f"{train_loss_log} | pose={train_metrics['pose']:.4f} "
        f"| pred-match p={pred_match_probability:.2f} "
        f"(actual {pred_match_batches / max(len(train_loader), 1):.2f})"
        f" | forest_sinkhorn={train_metrics['forest_sinkhorn']:.4f}"
    )

    improved = improved_adapt = improved_pose = False
    should_validate = (
        epoch == 1 or epoch % VAL_INTERVAL == 0 or epoch in (ADAPT_EPOCHS, ADAPT_EPOCHS + 1, FREEZE_VISION_EPOCH, FREEZE_VISION_EPOCH + 1, NUM_EPOCHS)
    )
    if should_validate:
        val_pred_metrics = validate(
            model, val_loader, device, OCC_THRESH, LAMBDA_OCC,
            LAMBDA_RADIUS, LAMBDA_CORR, LAMBDA_DEPTH_DELTA,
            LAMBDA_POSE, LAMBDA_WIDTH, pose_only=pose_objective_only, matching_mode="predicted",
            lambda_sinkhorn=0.0,
        )
        # GT matching is diagnostic; run it less often than end-to-end validation.
        run_gt_validation = epoch == 1 or epoch % 50 == 0
        val_gt_metrics = None
        if run_gt_validation:
            val_gt_metrics = validate(
                model, val_loader, device, OCC_THRESH, LAMBDA_OCC,
                LAMBDA_RADIUS, LAMBDA_CORR, LAMBDA_DEPTH_DELTA,
                LAMBDA_POSE, LAMBDA_WIDTH, pose_only=pose_objective_only, matching_mode="gt",
                lambda_sinkhorn=0.0,
            )
        # The primary val_* series and best_pose checkpoint track predicted matching.
        val_metrics = val_pred_metrics
        history["val_epoch"].append(epoch)
        history["val_stage"].append(stage)
        history["val_gt_pose"].append(val_gt_metrics["pose"] if val_gt_metrics is not None else float("nan"))
        history["val_pred_pose"].append(val_pred_metrics["pose"])
        history["val_gt_pose_valid_fraction"].append(
            val_gt_metrics["pose_valid_fraction"] if val_gt_metrics is not None else float("nan")
        )
        history["val_pred_pose_valid_fraction"].append(val_pred_metrics["pose_valid_fraction"])
        for key, metric in {
            "val_objective": "objective",
            "val_total": "total",
            "val_vision": "vision",
            "val_correspondence": "correspondence",
            "val_depth_structure": "depth_structure",
            "val_depth_delta": "depth_delta",
            "val_sinkhorn_reprojection": "sinkhorn_reprojection",
            "val_sinkhorn_matches_per_sample": "sinkhorn_matches_per_sample",
            "val_pose": "pose",
            "val_translation": "translation",
            "val_rotation": "rotation",
        }.items():
            history[key].append(val_metrics[metric])
        for component, weight in weights.items():
            history["val_" + component].append(val_metrics[component])
            history["val_" + component + "_weighted"].append(weight * val_metrics[component])
        gt_log = (
            f" | val_pose_GTmatch={val_gt_metrics['pose']:.4f} "
            f"(valid {val_gt_metrics['pose_valid_fraction']:.2f})"
            if val_gt_metrics is not None else ""
        )
        log += (
            f" | val_tot={val_metrics['total']:.4f} | val_vis={val_metrics['vision']:.4f} "
            f"| val_occ={val_metrics['vision_occupancy']:.4f} | val_rad={val_metrics['vision_radius']:.4f} "
            f"| val_depth={val_metrics['vision_depth']:.4f} | val_width={val_metrics['vision_width']:.4f} "
            f"{gt_log} "
            f"| val_pose_Predmatch={val_pred_metrics['pose']:.4f} "
            f"(valid {val_pred_metrics['pose_valid_fraction']:.2f})"
        )
        improved_pose = val_metrics["pose"] < best_pose_loss
        best_pose_loss = min(best_pose_loss, val_metrics["pose"])
        if pose_only:
            improved_adapt = val_metrics["pose"] < best_adapt_pose
            best_adapt_pose = min(best_adapt_pose, val_metrics["pose"])
        elif stage == "joint":
            improved = val_metrics["total"] < best_val_loss
            best_val_loss = min(best_val_loss, val_metrics["total"])
        if improved_pose:
            log += " | BEST POSE"
        if improved or improved_adapt:
            log += " | BEST ADAPT" if pose_only else " | BEST JOINT"
    state = {
        "epoch": epoch,
        "training_stage": stage,
        "model_state_dict": raw_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "args": {
            "data_dir": str(DATA_DIR), "output_dir": str(OUTPUT_DIR),
            "vision_checkpoint": str(VISION_CHECKPOINT), "pose_checkpoint": str(POSE_CHECKPOINT),
            "epochs": NUM_EPOCHS, "adapt_epochs": ADAPT_EPOCHS,
            "freeze_vision_epoch": FREEZE_VISION_EPOCH,
            "pred_match_ramp_epochs": PRED_MATCH_RAMP_EPOCHS,
            "pred_match_max_prob": PRED_MATCH_MAX_PROB,
            "adapt_lr": ADAPT_LEARNING_RATE, "lr": LEARNING_RATE,
            "pose_lr": POSE_LEARNING_RATE, "batch_size": BATCH_SIZE,
            "val_interval": VAL_INTERVAL, "seed": SEED,
            "num_workers": NUM_WORKERS, "img_size": IMG_SIZE,
            "patch_size": PATCH_SIZE, "embed_dim": EMBED_DIM,
            "depth": DEPTH, "num_heads": NUM_HEADS, "num_bins": NUM_BINS,
            "dropout": DROPOUT, "lambda_occ": LAMBDA_OCC,
            "lambda_radius": LAMBDA_RADIUS, "occ_thresh": OCC_THRESH,
            "lambda_corr": LAMBDA_CORR, "lambda_depth_delta": LAMBDA_DEPTH_DELTA,
            "lambda_pose": LAMBDA_POSE, "lambda_width": LAMBDA_WIDTH,
            "pose_embed_dim": POSE_EMBED_DIM, "pose_num_heads": POSE_NUM_HEADS,
            "pose_depth": POSE_DEPTH, "pose_dropout": POSE_DROPOUT,
            "fov_degrees": FOV_DEGREES, "radius_scale": RADIUS_SCALE,
            "distance_scale": DISTANCE_SCALE,
        },
        "history": history,
        "best_val_loss": best_val_loss,
        "best_adapt_pose": best_adapt_pose,
        "best_pose_loss": best_pose_loss,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
        },
    }
    torch.save(state, os.path.join(OUTPUT_DIR, "checkpoints", "latest.pt"))
    if improved:
        torch.save(state, os.path.join(OUTPUT_DIR, "checkpoints", "best.pt"))
    if improved_adapt:
        torch.save(state, os.path.join(OUTPUT_DIR, "checkpoints", "best_adapt.pt"))
    if improved_pose:
        torch.save(state, BEST_POSE_PATH)
    with open(os.path.join(OUTPUT_DIR, "history.json"), "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(log + f" | time={time.perf_counter() - epoch_start:.1f}s", flush=True)

# Keep final weights available to the notebook's inference cells.
latest_model_state_dict = {key: value.detach().cpu().clone() for key, value in raw_model.state_dict().items()}



Stage: pose_adapt; vision lr=0; pose lr=0.0001
Epoch 1 [pose_adapt]: obj=0.3569 | pose=0.1756 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0564 | val_tot=4.2630 | val_vis=2.3032 | val_occ=0.1272 | val_rad=0.0530 | val_depth=0.4605 | val_width=0.4065  | val_pose_GTmatch=0.0759 (valid 1.00) | val_pose_Predmatch=0.4163 (valid 0.94) | BEST POSE | BEST ADAPT | time=333.8s
Epoch 2 [pose_adapt]: obj=0.2847 | pose=0.1395 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0569 | time=283.2s
Epoch 3 [pose_adapt]: obj=0.2495 | pose=0.1220 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0552 | time=298.1s
Epoch 4 [pose_adapt]: obj=0.1541 | pose=0.0741 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0590 | time=276.4s
Epoch 5 [pose_adapt]: obj=0.1260 | pose=0.0604 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0517 | time=283.2s
Epoch 6 [pose_adapt]: obj=0.0982 | pose=0.0463 | pred-match p=0.00 (actual 0.00) | forest_sinkhorn=0.0552 | time=276.8s
Epoch 7 [pose_adapt]

KeyboardInterrupt: 

## Load saved training state

Load fine-tuning weights and history for plots/inference. Training resume is controlled by `RESUME_TRAINING` and handled in the model setup cell.


In [ ]:
latest_checkpoint_path = Path(LATEST_MODEL_PATH)
best_checkpoint_path = Path(BEST_MODEL_PATH)
best_pose_path = Path(BEST_POSE_PATH)

checkpoint_path = best_pose_path
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    raw_model.load_state_dict(checkpoint["model_state_dict"])
    history = checkpoint["history"]
    best_val_loss = checkpoint.get("best_val_loss", float("inf"))
    best_epoch = checkpoint.get("epoch")
    print(f"Loaded epoch {best_epoch}, stage {checkpoint.get('training_stage')}: {checkpoint_path}")
else:
    print(f"No joint checkpoint at {checkpoint_path}; it is saved during the joint stage.")


In [ ]:
if False:
    val_epochs = np.array(history["val_epoch"])
    plt.figure(figsize=(10, 5))
    plt.plot(val_epochs, history["val_translation"], ".-", label="Translation loss")
    plt.plot(val_epochs, history["val_rotation"], ".-", label="Yaw loss")
    plt.axvline(ADAPT_EPOCHS + 0.5, color="gray", linestyle=":", label="Joint stage starts")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Training plots

In [ ]:
checkpoint_path = Path(LATEST_MODEL_PATH)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
raw_model.load_state_dict(checkpoint["model_state_dict"])
history = checkpoint["history"]
best_val_loss = checkpoint.get("best_val_loss", float("inf"))
best_epoch = checkpoint.get("epoch")
latest_model_state_dict = {key: value.detach().cpu().clone() for key, value in raw_model.state_dict().items()}


In [ ]:
epochs = np.array(history["epoch"])
val_epochs = np.array(history["val_epoch"])

plt.figure(figsize=(11, 6))

# Train
plt.plot(history["epoch"], history["vision_loss"], label="Train vision")
plt.plot(history["epoch"], history["correspondence_loss"], label="Train correspondence")
plt.plot(history["epoch"], history["depth_structure_loss"], label="Train depth structure")
plt.plot(history["epoch"], history["depth_delta_loss"], label="Train matched depth delta")
plt.plot(history["epoch"], history["pose_loss"], label="Train pose")


# Validation
plt.plot(history["val_epoch"], history["val_vision"], "o--", label="Val vision")
plt.plot(history["val_epoch"], history["val_correspondence"], "o--", label="Val correspondence")
plt.plot(history["val_epoch"], history["val_depth_structure"], "o--", label="Val depth structure")
plt.plot(history["val_epoch"], history["val_depth_delta"], "o--", label="Val matched depth delta")
plt.plot(history["val_epoch"], history["val_gt_pose"], "o--", label="Val pose (GT matches)")
plt.plot(history["val_epoch"], history["val_pred_pose"], "o--", label="Val pose (predicted matches)")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Raw loss (log scale)")
plt.title("Training and validation loss components")
plt.grid(True, alpha=0.3)
if ADAPT_EPOCHS > 0:
    plt.axvline(ADAPT_EPOCHS + 0.5, color="gray", linestyle=":", label="Joint stage starts")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
forest_sinkhorn = np.asarray(history.get("forest_sinkhorn_loss", []), dtype=float)
train_epochs = np.asarray(history.get("epoch", []))
if len(forest_sinkhorn) == len(train_epochs):
    ax.plot(train_epochs, forest_sinkhorn, label="Forest Sinkhorn")
if ADAPT_EPOCHS > 0:
    ax.axvline(ADAPT_EPOCHS + 0.5, color="gray", linestyle=":", label="Joint stage starts")
ax.set_xlabel("Epoch")
ax.set_ylabel("Sinkhorn reprojection loss")
ax.set_title("Forest Sinkhorn loss")
ax.grid(True, alpha=0.3)
if ax.lines:
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Separate vision-loss components. False = raw losses; True = actual
# weighted contributions to vision_loss (weights recorded at each epoch).
PLOT_WEIGHTED_VISION = False
suffix = "_weighted" if PLOT_WEIGHTED_VISION else ""
components = [
    ("vision_occupancy", "Occupancy"),
    ("vision_radius", "Radius"),
    ("vision_depth", "Depth"),
    ("vision_width", "Width"),
]
train_epochs = np.asarray(history.get("epoch", []))
validation_epochs = np.asarray(history.get("val_epoch", []))

available = any(
    np.isfinite(np.asarray(history.get(prefix + key + suffix, []), dtype=float)).any()
    for key, _ in components
    for prefix in ("", "val_")
)
if available:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (key, label) in zip(axes.flat, components):
        for prefix, xs, style, legend in (
            ("", train_epochs, "-", "Train"),
            ("val_", validation_epochs, "o--", "Validation"),
        ):
            values = np.asarray(history.get(prefix + key + suffix, []), dtype=float)
            if len(values) == len(xs) and np.isfinite(values).any():
                ax.plot(xs, values, style, label=legend)
        ax.set_title(label)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Weighted contribution" if PLOT_WEIGHTED_VISION else "Raw loss")
        if ADAPT_EPOCHS > 0:
            ax.axvline(ADAPT_EPOCHS + 0.5, color="gray", linestyle=":", label="Joint stage starts")
        ax.grid(True, alpha=0.3)
        if ax.lines:
            ax.legend()
    fig.suptitle("Vision loss components — " + ("weighted" if PLOT_WEIGHTED_VISION else "raw"))
    plt.tight_layout()
    plt.show()
else:
    print("This history has no separate vision components yet. They are recorded from the next training epoch.")


## Inference on test data

In [ ]:
# Choose which trained weights to inspect.
INFERENCE_MODEL = INFERENCE_MODEL.lower()
INFERENCE_MODEL = "best_pose"
paths = {"best": BEST_MODEL_PATH, "best_pose": BEST_POSE_PATH, "latest": LATEST_MODEL_PATH, "best_adapt": BEST_ADAPT_PATH}
if INFERENCE_MODEL not in paths:
    raise ValueError("Choose 'best', 'best_pose', 'latest' or 'best_adapt'")
checkpoint_path = Path(paths[INFERENCE_MODEL])
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
raw_model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("INFERENCE_MODEL:", INFERENCE_MODEL)
print("Epoch:", checkpoint.get("epoch"), "Stage:", checkpoint.get("training_stage"))

test_dataset = SceneTwoPairsDataset(
    root_dir=os.path.join(DATA_DIR, "testdataset"),
    image_size=IMG_SIZE,
    return_two_pairs=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=INFERENCE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
batch = next(iter(test_loader))
img_a, vision_a, img_b, vision_b, pose_ab = (x.to(device) for x in batch)
with torch.no_grad():
    pred_vision, pred_vision_b, pred_pose, pose_info = model(
        img_a,
        img_b,
        target_vision_a=vision_a,
        target_vision_b=vision_b,
        return_pose_inputs=True,
        occ_thresh=OCC_THRESH,
    )


## Vision prediction plots

In [ ]:
sample_idx = VISION_SAMPLE_IDX +2
bins = np.arange(pred_vision.shape[1])
PLOT_OCC_THRESHOLD = 0.4

if INFERENCE_SHOW_IMAGES:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    names = ["occupancy", "radius", "depth"]

    gt_occ = vision_a[sample_idx, :, 0].detach().cpu().numpy()
    pr_occ = pred_vision[sample_idx, :, 0].detach().cpu().numpy()

    for j, ax in enumerate(axes):
        gt = vision_a[sample_idx, :, j].detach().cpu().numpy()
        pr = pred_vision[sample_idx, :, j].detach().cpu().numpy()

        ax.plot(bins, gt, label="GT", color="tab:blue")

        if j == 0:
            ax.plot(bins, pr, label="Pred", color="tab:orange")
        else:
            pr_mask = pr_occ > PLOT_OCC_THRESHOLD
            ax.scatter(
                bins[pr_mask],
                pr[pr_mask],
                label="Pred",
                color="tab:orange",
                s=25,
            )

        ax.set_ylabel(names[j])
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[-1].set_xlabel("Bin")
    fig.suptitle(f"Vision prediction sample {sample_idx}")
    plt.tight_layout()
    plt.show()

    fig_cyl, axes_cyl = plot_estimated_cylinders_on_images(
        [img_a[sample_idx], img_b[sample_idx]],
        [pred_vision[sample_idx], pred_vision_b[sample_idx]],
        titles=["Image A: pred cylinders", "Image B: pred cylinders"],
        occ_threshold=PLOT_OCC_THRESHOLD,
        flip_x=False,
    )
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img_a[sample_idx].cpu().permute(1, 2, 0))
    axes[0].set_title("Image A")
    axes[0].axis("off")
    axes[1].imshow(img_b[sample_idx].cpu().permute(1, 2, 0))
    axes[1].set_title("Image B")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    if pose_info["pose_valid"][sample_idx] and pose_ab[sample_idx, :2].norm() > 1e-6:
        plot_relative_pose(pose_ab[sample_idx], pred_pose[sample_idx])
    else:
        print("Selected sample has no evaluable pose; choose another VISION_SAMPLE_IDX.")


In [ ]:
from metrics import evaluate_vision

vision_metrics = evaluate_vision(model, test_loader, device, occ_threshold=0.3)


In [ ]:
from metrics import evaluate_pose

# 1. Referens: GT-bin och GT-matchningar.
metrics_gt = evaluate_pose(
    model, test_loader, device,
    matching_mode="gt",
)

# 2. GT anger cylinderpositionerna; matchningen skattas.
metrics_gt_bins = evaluate_pose(
    model, test_loader, device,
    matching_mode="gt_bins",
)

# 3. Hela kedjan från bilder, utan GT-cylinderdata.
metrics_predicted = evaluate_pose(
    model, test_loader, device,
    matching_mode="predicted",
    pred_occ_threshold=0.2,
)


## Forest

In [ ]:
# Evaluate one or more forest datasets independently. Each stride gets its own
# loader and metric result, so the different motion baselines are not mixed.
from torchvision import transforms

forest_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

if not FOREST_EVAL_PRESETS:
    raise ValueError("FOREST_EVAL_PRESETS must contain at least one dataset preset")

forest_eval_loaders = {}
forest_eval_datasets = {}
for preset in FOREST_EVAL_PRESETS:
    if preset not in FOREST_DATASET_PRESETS:
        raise ValueError(f"Unknown forest evaluation preset: {preset}")
    dataset_dir, poses_file = FOREST_DATASET_PRESETS[preset]
    dataset = ForestPairDataset(
        dataset_dir=dataset_dir,
        poses_file=poses_file,
        transform=forest_transform,
        return_names=False,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )
    forest_eval_datasets[preset] = dataset
    forest_eval_loaders[preset] = loader
    print(f"Forest evaluation {preset}: {len(dataset)} pairs from {dataset_dir / poses_file}")

# Used by the visualization cell below.
FOREST_VISUALIZATION_PRESET = FOREST_EVAL_PRESETS[0]
forest_dataset = forest_eval_datasets[FOREST_VISUALIZATION_PRESET]
forest_loader = forest_eval_loaders[FOREST_VISUALIZATION_PRESET]


In [ ]:
from metrics import evaluate_pose_forest

forest_metrics_by_preset = {}
for preset, loader in forest_eval_loaders.items():
    print(f"\nEvaluating forest preset: {preset}")
    forest_metrics_by_preset[preset] = evaluate_pose_forest(
        model,
        loader,
        device,
        predict_pose=lambda m, a, b: m.predict_pose_from_images(a, b),
    )

# Backwards-friendly alias for the selected visualization preset.
forest_metrics = forest_metrics_by_preset[FOREST_VISUALIZATION_PRESET]


In [ ]:
if FOREST_ENABLED:
    forest_sample_idx = FOREST_SAMPLE_IDX
    model.eval()
    forest_batch = next(iter(forest_loader))
    img_a_forest, img_b_forest, _ = forest_batch

    img_a_forest = img_a_forest.to(device)
    img_b_forest = img_b_forest.to(device)

    with torch.no_grad():
        pred_vision_a_forest, pred_vision_b_forest, pred_pose_forest = model(
            img_a_forest, img_b_forest, compute_pose=False
        )

    fig_forest_cyl, axes_forest_cyl = plot_estimated_cylinders_on_images(
        [
            img_a_forest[forest_sample_idx],
            img_b_forest[forest_sample_idx],
        ],
        [
            pred_vision_a_forest[forest_sample_idx],
            pred_vision_b_forest[forest_sample_idx],
        ],
        titles=["Forest A: pred cylinders", "Forest B: pred cylinders"],
        occ_threshold=FOREST_OCC_THRESHOLD,
        flip_x=False,
    )
    plt.show()

    print("forest pred_vision_a shape:", pred_vision_a_forest.shape)
    print("Forest visualization is vision only: no GT bin matches are available.")


In [ ]:
from torchvision import transforms
from forest_dataset import ForestPairDataset
from metrics import evaluate_pose_forest

skrylle_dir = DATA_DIR / "forestdataset" / "snoge"
strides = [1, 5, 10, 20]

skrylle_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

skrylle_metrics_by_stride = {}

for stride in strides:
    poses_file = f"relative_poses_stride_{stride}.json"

    dataset = ForestPairDataset(
        dataset_dir=skrylle_dir,
        poses_file=poses_file,
        transform=skrylle_transform,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == "cuda",
    )

    print(f"\nEvaluating Snoge, stride {stride} ({len(dataset)} pairs)")
    skrylle_metrics_by_stride[stride] = evaluate_pose_forest(
        model,
        loader,
        device,
        predict_pose=lambda m, a, b: m.predict_pose_from_images(a, b),
    )